# Module 8: Async Task Queue with Kafka

本章目标：
- 理解消息队列作为任务队列的设计模式
- 用 aiokafka 从零构建一个分布式异步任务队列
- 了解 Taskiq 框架及其与 Kafka 的集成方式
- 用 Taskiq `InMemoryBroker` 演示任务系统的核心概念

---

## 前置条件

- 集群运行中：`docker compose up -d`（根目录）
- 已安装依赖：`uv sync`
- Module 8 额外依赖：`uv sync --extra taskiq`

## 为什么用 Kafka 做任务队列？

```
传统任务队列（Redis/RabbitMQ）：
  - 任务消费后即删除
  - 历史无法重放

Kafka 作为任务队列：
  - 任务消息持久化存储（可按配置保留天数）
  - 可重放失败任务
  - 天然支持多 Worker 组并行处理
  - 吞吐量远高于传统队列
```

**典型场景：**
- 大规模图像/视频处理
- 批量邮件/通知发送
- 数据 ETL 流水线
- 事件驱动的微服务通信

## Part 1: 用 aiokafka 手搓任务队列

理解任务队列的本质：
- **生产者** = 任务提交方（发送 JSON 任务到 Kafka）
- **消费者** = Worker（从 Kafka 读取并执行任务）
- **任务消息** = 包含任务名称、参数、元数据的 JSON 消息

In [ ]:
import asyncio
import json
import time
import uuid
from aiokafka import AIOKafkaProducer, AIOKafkaConsumer
from aiokafka.admin import AIOKafkaAdminClient, NewTopic

BROKERS = "localhost:19094,localhost:29094,localhost:39094"
TASK_TOPIC = "task-queue"
RESULT_TOPIC = "task-results"

In [ ]:
async def setup_task_topics(brokers):
    admin = AIOKafkaAdminClient(bootstrap_servers=brokers)
    await admin.start()
    try:
        existing = await admin.list_topics()
        new_topics = [
            NewTopic(name=t, num_partitions=3, replication_factor=3)
            for t in [TASK_TOPIC, RESULT_TOPIC] if t not in existing
        ]
        if new_topics:
            await admin.create_topics(new_topics)
            await asyncio.sleep(1)
            print(f"✓ Created topics: {[t.name for t in new_topics]}")
        else:
            print("Topics already exist")
    finally:
        await admin.close()

await setup_task_topics(BROKERS)

### 1.1 定义任务类型与序列化格式

In [ ]:
# 任务消息格式
def make_task(task_name: str, **kwargs) -> bytes:
    return json.dumps({
        "task_id": str(uuid.uuid4())[:8],
        "task": task_name,
        "args": kwargs,
        "submitted_at": time.time(),
    }).encode()

# 定义实际任务处理函数
TASK_REGISTRY = {}

def task(fn):
    """装饰器：将函数注册为可调用任务。"""
    TASK_REGISTRY[fn.__name__] = fn
    return fn

@task
async def send_email(to: str, subject: str, body: str) -> dict:
    await asyncio.sleep(0.1)  # 模拟发送延迟
    return {"status": "sent", "to": to, "chars": len(body)}

@task
async def resize_image(url: str, width: int, height: int) -> dict:
    await asyncio.sleep(0.2)  # 模拟处理延迟
    return {"status": "done", "output": f"{url}-{width}x{height}.jpg"}

@task
async def calculate(expression: str) -> dict:
    import math
    safe_env = {"__builtins__": {}, "math": math, "sum": sum, "range": range, "len": len}
    result = eval(expression, safe_env)
    return {"result": result}

print(f"Registered tasks: {list(TASK_REGISTRY.keys())}")

### 1.2 Task Producer：提交任务

In [ ]:
async def submit_tasks(brokers, topic, tasks: list):
    producer = AIOKafkaProducer(bootstrap_servers=brokers)
    await producer.start()
    try:
        for task_name, kwargs in tasks:
            msg = make_task(task_name, **kwargs)
            task_data = json.loads(msg)
            meta = await producer.send_and_wait(topic, msg)
            print(f"  [Submit] {task_data['task']}({kwargs}) -> task_id={task_data['task_id']}, partition={meta.partition}")
    finally:
        await producer.stop()

tasks_to_submit = [
    ("send_email", {"to": "alice@example.com", "subject": "Welcome", "body": "Hello Alice!"}),
    ("send_email", {"to": "bob@example.com", "subject": "Update", "body": "Check this out"}),
    ("resize_image", {"url": "s3://bucket/photo.jpg", "width": 800, "height": 600}),
    ("resize_image", {"url": "s3://bucket/avatar.png", "width": 128, "height": 128}),
    ("calculate", {"expression": "2 ** 10"}),
    ("calculate", {"expression": "sum(range(100))"}),
]

print("=== Submitting tasks ===")
await submit_tasks(BROKERS, TASK_TOPIC, tasks_to_submit)

### 1.3 Task Worker：消费并执行任务

In [ ]:
async def run_worker(brokers, task_topic, result_topic, worker_id, max_tasks=6):
    consumer = AIOKafkaConsumer(
        task_topic,
        bootstrap_servers=brokers,
        group_id="worker-pool",
        auto_offset_reset="earliest",
        enable_auto_commit=False,
    )
    result_producer = AIOKafkaProducer(bootstrap_servers=brokers)
    await consumer.start()
    await result_producer.start()
    processed = 0
    try:
        while processed < max_tasks:
            results = await consumer.getmany(timeout_ms=2000, max_records=2)
            if not results:
                break
            for tp_msgs in results.values():
                for msg in tp_msgs:
                    task_data = json.loads(msg.value)
                    task_fn = TASK_REGISTRY.get(task_data["task"])
                    start = time.perf_counter()
                    if task_fn:
                        try:
                            result = await task_fn(**task_data["args"])
                            status = "success"
                        except Exception as e:
                            result = {"error": str(e)}
                            status = "failed"
                    else:
                        result = {"error": f"unknown task: {task_data['task']}"}
                        status = "failed"

                    elapsed_ms = (time.perf_counter() - start) * 1000
                    result_msg = json.dumps({
                        "task_id": task_data["task_id"],
                        "task": task_data["task"],
                        "status": status,
                        "result": result,
                        "worker": worker_id,
                        "elapsed_ms": round(elapsed_ms, 1),
                    }).encode()
                    await result_producer.send_and_wait(result_topic, result_msg)
                    await consumer.commit()
                    print(f"  [{worker_id}] {task_data['task']}({task_data['task_id']}) → {status} in {elapsed_ms:.0f}ms")
                    processed += 1
    finally:
        await consumer.stop()
        await result_producer.stop()
    return processed

print("=== Running 2 workers in parallel ===")
w1, w2 = await asyncio.gather(
    run_worker(BROKERS, TASK_TOPIC, RESULT_TOPIC, "worker-1", max_tasks=6),
    run_worker(BROKERS, TASK_TOPIC, RESULT_TOPIC, "worker-2", max_tasks=6),
)
print(f"\nWorker-1 processed: {w1} tasks")
print(f"Worker-2 processed: {w2} tasks")

### 1.4 查看任务结果

In [ ]:
from aiokafka import TopicPartition

async def show_results(brokers, result_topic):
    admin = AIOKafkaAdminClient(bootstrap_servers=brokers)
    await admin.start()
    descriptions = await admin.describe_topics([result_topic])
    num_partitions = len(descriptions[0]['partitions'])
    await admin.close()

    consumer = AIOKafkaConsumer(
        bootstrap_servers=brokers,
        auto_offset_reset="earliest",
    )
    tps = [TopicPartition(result_topic, p) for p in range(num_partitions)]
    consumer.assign(tps)
    await consumer.start()
    try:
        results = []
        while True:
            batch = await consumer.getmany(*tps, timeout_ms=1000, max_records=50)
            if not batch:
                break
            for tp_msgs in batch.values():
                for msg in tp_msgs:
                    results.append(json.loads(msg.value))
    finally:
        await consumer.stop()

    print(f"\n=== Task Results ({len(results)} total) ===")
    for r in results:
        print(f"  [{r['status']:7s}] {r['task']}({r['task_id']}) → {r['result']} (worker={r['worker']}, {r['elapsed_ms']}ms)")

await show_results(BROKERS, RESULT_TOPIC)

## Part 2: Taskiq 框架简介

[Taskiq](https://taskiq-python.github.io/) 是一个现代化的 Python 异步任务队列框架，支持多种 Broker 后端。

### 核心概念

| 概念 | 说明 |
|------|------|
| `broker` | 消息后端（InMemory/Redis/RabbitMQ/...）|
| `@broker.task` | 将函数注册为分布式任务 |
| `task.kiq(args)` | 异步提交任务（Kick In Queue）|
| `taskiq worker` | 启动 Worker 进程 |

In [ ]:
# Part 2: Taskiq 基础演示（使用 InMemoryBroker，无需外部依赖）
try:
    from taskiq import InMemoryBroker
    HAS_TASKIQ = True
except ImportError:
    print("安装 taskiq：uv sync --extra taskiq")
    HAS_TASKIQ = False

print(f"Taskiq available: {HAS_TASKIQ}")

In [ ]:
if HAS_TASKIQ:
    from taskiq import InMemoryBroker

    broker = InMemoryBroker()

    @broker.task
    async def process_order(order_id: str, amount: float) -> dict:
        await asyncio.sleep(0.05)
        return {"order_id": order_id, "charged": amount, "status": "completed"}

    @broker.task
    async def generate_report(report_type: str, date: str) -> dict:
        await asyncio.sleep(0.1)
        return {"report_type": report_type, "date": date, "rows": 1024}

    async def taskiq_demo():
        await broker.startup()
        print("=== Taskiq InMemoryBroker Demo ===")

        # 提交任务
        t1 = await process_order.kiq("ORD-001", 99.99)
        t2 = await process_order.kiq("ORD-002", 49.50)
        t3 = await generate_report.kiq("monthly", "2026-04")
        print(f"  Submitted 3 tasks")

        # 等待结果
        r1 = await t1.wait_result(timeout=5)
        r2 = await t2.wait_result(timeout=5)
        r3 = await t3.wait_result(timeout=5)

        for task_result in [r1, r2, r3]:
            print(f"  Result: {task_result.return_value}")

        await broker.shutdown()

    await taskiq_demo()

## Part 3: 将 Taskiq 接入 Kafka

Taskiq 的 Broker 是可插拔的。要使用 Kafka 作为后端，可以实现自定义 Broker 类，
继承 `taskiq.AsyncBroker` 并在内部使用 `aiokafka`：

```python
# 示例：自定义 Kafka Broker（需要自行实现）
from taskiq import AsyncBroker, AsyncResultBackend
from aiokafka import AIOKafkaProducer, AIOKafkaConsumer

class KafkaBroker(AsyncBroker):
    def __init__(self, bootstrap_servers: str, topic: str = "taskiq-tasks"):
        super().__init__()
        self.bootstrap_servers = bootstrap_servers
        self.topic = topic
        self._producer = None

    async def startup(self):
        self._producer = AIOKafkaProducer(bootstrap_servers=self.bootstrap_servers)
        await self._producer.start()

    async def shutdown(self):
        if self._producer:
            await self._producer.stop()

    async def kick(self, message):
        await self._producer.send_and_wait(
            self.topic,
            message.model_dump_json().encode()
        )

    async def listen(self):
        consumer = AIOKafkaConsumer(
            self.topic,
            bootstrap_servers=self.bootstrap_servers,
            group_id="taskiq-workers",
        )
        await consumer.start()
        async for msg in consumer:
            yield msg.value
```

然后像普通 Taskiq 一样使用：

```python
broker = KafkaBroker("localhost:19094,localhost:29094,localhost:39094")

@broker.task
async def my_task(x: int) -> int:
    return x * 2

# 提交任务
await my_task.kiq(42)

# 启动 Worker（命令行）
# taskiq worker module_name:broker
```

## 总结

| 方式 | 优点 | 适用场景 |
|------|------|----------|
| 纯 aiokafka DIY | 完全控制、无额外依赖 | 需要深度定制的场景 |
| Taskiq + InMemoryBroker | 开发调试快速 | 单机测试 |
| Taskiq + 自定义 KafkaBroker | 生产级分布式任务 | 高吞吐、可重放 |

**Kafka 作为任务队列的优势：**
- **持久化**：任务不会因 Worker 宕机而丢失
- **可重放**：失败任务可以从任意 offset 重新执行
- **水平扩展**：增加 Worker 即可提升处理能力
- **可观测**：Kafka 的 lag 监控直接反映任务积压情况

---

至此，完成了 Kafka Python 教程的全部 8 个模块！

**推荐延伸学习：**
- [Kafka Streams](https://kafka.apache.org/documentation/streams/) — 流处理
- [ksqlDB](https://ksqldb.io/) — SQL 查询 Kafka 数据
- [Faust](https://faust-streaming.github.io/faust/) — Python 流处理框架
- [Taskiq 官方文档](https://taskiq-python.github.io/)